In [ ]:
import os
import sys

if sys.platform == "linux":
    os.environ["MUJOCO_GL"] = "egl"
    os.environ["PYOPENGL_PLATFORM"] = "egl"

# Create Simulation Environment

In [ ]:
import mujoco
from mujoco import MjModel, MjData, MjSpec
from scipy.spatial.transform import Rotation as R
import numpy as np

from molmo_spaces.configs.robot_configs import FrankaRobotConfig
from molmo_spaces.robots.robot_views.franka_droid_view import FrankaDroidRobotView
from molmo_spaces.molmo_spaces_constants import get_procthor_10k_houses, get_robot_path
from molmo_spaces.robots.franka import FrankaRobot
from molmo_spaces.utils.lazy_loading_utils import install_scene_with_objects_and_grasps_from_path, install_uid

In [ ]:
houses = get_procthor_10k_houses(split="val")
house_xml_path = houses["val"][0]["base"]
install_scene_with_objects_and_grasps_from_path(house_xml_path)

spec = MjSpec.from_file(house_xml_path)

robot_config = FrankaRobotConfig(base_size=[0.5, 0.5, 0.75])
robot_file_path = get_robot_path(robot_config.name) / robot_config.robot_xml_path
robot_spec = MjSpec.from_file(str(robot_file_path))

FrankaRobot.add_robot_to_scene(
    robot_config,
    spec,
    robot_spec,
    prefix=robot_config.robot_namespace,
    pos=[6.8, 9.75],
    quat=R.from_euler("z", 90, degrees=True).as_quat(scalar_first=True),
)

spec.camera(robot_config.robot_namespace + "gripper/wrist_camera").resolution = [640, 360]
spec.body(robot_config.robot_namespace + "fr3_link0").add_camera(
    pos=[0.1, 0.57, 0.66],
    quat=[-0.3633, -0.1241, 0.4263, 0.8191],
    fovy=71.0,
    resolution=[640, 360],
    name="robot_0/exo_camera_1",
)

bowl_xml_path = install_uid("Bowl_3")
bowl_spec = MjSpec.from_file(str(bowl_xml_path))
root_body = bowl_spec.worldbody.first_body()
receptacle_frame = spec.worldbody.add_frame(
    pos=[7.1, 10.2, 1.01],
    quat=R.from_euler("x", 90, degrees=True).as_quat(scalar_first=True)
)
receptacle_frame.attach_body(root_body, prefix="place_receptacle/")

model: MjModel = spec.compile()
data = MjData(model)
view = FrankaDroidRobotView(data, robot_config.robot_namespace)

view.set_qpos_dict(robot_config.init_qpos)
mujoco.mj_forward(model, data)
for mg_id in view.move_group_ids():
    mg = view.get_move_group(mg_id)
    mg.ctrl = mg.noop_ctrl
mujoco.mj_forward(model, data)

renderer = mujoco.Renderer(model, 360, 640)
scene_option = mujoco.MjvOption()
scene_option.sitegroup = 0

def render():
    renderer.update_scene(data, camera="robot_0/exo_camera_1", scene_option=scene_option)
    exo_img = renderer.render()
    renderer.update_scene(data, camera="robot_0/gripper/wrist_camera", scene_option=scene_option)
    cam_img = renderer.render()
    return {
        "exo_camera_1": exo_img,
        "wrist_camera": cam_img,
    }

## Sample render

This is what the policy sees!

In [ ]:
from PIL import Image

images = render()
stacked_img = np.hstack([images["exo_camera_1"], images["wrist_camera"]])
Image.fromarray(stacked_img)

# Load Model

In [ ]:
from huggingface_hub import snapshot_download
from olmo.eval.configure_real_robot import RealRobotVLAPolicy, RealRobotVLAPolicyConfig

ckpt_path = snapshot_download("allenai/MolmoBot-DROID")

class MockConfig:
    def __init__(self, policy_config):
        self.policy_config = policy_config

policy_config = RealRobotVLAPolicyConfig()
policy_config.checkpoint_path = ckpt_path
policy_config.action_type = "joint_pos"
policy_config.action_keys["arm"] = "joint_pos"

mock_config = MockConfig(policy_config)

policy = RealRobotVLAPolicy(config=mock_config, task_type="manipulation")

# Run Policy

In [ ]:
from tqdm import tqdm
from moviepy.video.io.ImageSequenceClip import ImageSequenceClip
from IPython.display import Video


task = "put the salt shaker in the bowl"

policy_dt_ms = 66
episode_dur = 30.0

frames = []
for _ in tqdm(range(round(episode_dur * 1000 / policy_dt_ms))):
    jp = view.get_move_group("arm").joint_pos
    gripper_input = view.get_move_group("gripper").joint_pos
    obs = {
        "task": task,
        "qpos": {
            "arm": jp,
            "gripper": gripper_input,
        },
        **render()
    }

    frame = np.hstack([obs["exo_camera_1"], obs["wrist_camera"]])
    frames.append(frame)

    action = policy.get_action(obs)

    for mg_id in action.keys():
        view.get_move_group(mg_id).ctrl = action[mg_id]

    mujoco.mj_step(model, data, nstep=policy_dt_ms // round(model.opt.timestep * 1000))
    

video = ImageSequenceClip(frames, fps=15)
video.write_videofile("rollout.mp4", audio=False)

Video("rollout.mp4")